# Classificação Supervisionada com Gustafson-Kessel (GK) no Dataset Adult (Balanceado)

Este notebook aplica o balanceamento das classes do dataset Adult por oversampling da classe minoritária antes do treinamento do GK supervisionado. O objetivo é comparar o desempenho do agrupamento com e sem desbalanceamento.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, confusion_matrix, mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.utils import resample
import os
import seaborn as sns
os.makedirs('img', exist_ok=True)

## 1. Carregamento e Balanceamento dos Dados

In [ ]:
adult_df_raw = pd.read_csv('data/AdultDataset/adult.data', header=None, na_values=' ?', skipinitialspace=True)
adult_df_raw.columns = [
    'age', 'workclass', 'fnlwgt', 'education', 'education-num', 'marital-status',
    'occupation', 'relationship', 'race', 'sex', 'capital-gain', 'capital-loss',
    'hours-per-week', 'native-country', 'income'
]
adult_df_raw['income_bin'] = adult_df_raw['income'].apply(lambda val: 1 if val.strip() == '>50K' else 0)
adult_df = adult_df_raw.dropna().copy()
df_majority = adult_df[adult_df['income_bin'] == 0]
df_minority = adult_df[adult_df['income_bin'] == 1]
n_majority = len(df_majority)
df_minority_upsampled = resample(
    df_minority,
    replace=True,
    n_samples=n_majority,
    random_state=42
)
adult_df_balanced = pd.concat([df_majority, df_minority_upsampled])
adult_df_balanced = adult_df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)
print('Distribuição após balanceamento:')
print(adult_df_balanced['income_bin'].value_counts())

## 2. Pré-processamento dos Dados

In [ ]:
for col in adult_df_balanced.select_dtypes(include='object').columns:
    if col != 'income':
        adult_df_balanced[col] = LabelEncoder().fit_transform(adult_df_balanced[col].astype(str))
X = adult_df_balanced.drop(['income', 'income_bin'], axis=1).values
y = adult_df_balanced['income_bin'].values
scaler = StandardScaler()
X = scaler.fit_transform(X)
plt.figure(figsize=(5,3))
sns.countplot(x=y)
plt.title('Distribuição da coluna income (binária) após balanceamento')
plt.xticks([0,1],["<=50K",">50K"])
plt.show()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y)

In [ ]:
n_clusters = 2

In [ ]:
class GKSupervisionado:
    def __init__(self, n_clusters=2, m=2.0, max_iter=50, tol=1e-5, random_state=0):
        self.n_clusters = n_clusters
        self.m = m
        self.max_iter = max_iter
        self.tol = tol
        self.random_state = random_state
        self.cluster_labels_ = None
        self.U_ = None
        self.V_ = None
        self.F_ = None

    def fit(self, X, y):
        np.random.seed(self.random_state)
        n_samples, n_features = X.shape
        U = np.random.dirichlet(np.ones(self.n_clusters), size=n_samples).T
        q = self.m
        for _ in range(self.max_iter):
            V = (U ** q) @ X / np.sum(U ** q, axis=1)[:, None]
            F = np.zeros((self.n_clusters, n_features, n_features))
            for i in range(self.n_clusters):
                diff = X - V[i]
                um = (U[i] ** q)[:, None]
                F[i] = (um * diff).T @ diff / np.sum(um)
                F[i] += np.eye(n_features) * 1e-6
            A = np.zeros_like(F)
            for i in range(self.n_clusters):
                detF = np.linalg.det(F[i])
                if detF <= 0:
                    detF = 1e-6
                A[i] = (detF ** (1 / n_features)) * np.linalg.inv(F[i])
            D = np.zeros((self.n_clusters, n_samples))
            for i in range(self.n_clusters):
                diff = X - V[i]
                D[i] = np.einsum('ij,jk,ik->i', diff, A[i], diff)
            for i in range(self.n_clusters):
                denom = np.sum((D[i][:, None] / D.T) ** (1 / (q - 1)), axis=1)
                U[i] = 1.0 / denom
        self.U_ = U
        self.V_ = V
        self.F_ = F
        clusters = np.argmax(U, axis=0)
        self.cluster_labels_ = []
        for i in range(self.n_clusters):
            mask = (clusters == i)
            if np.any(mask):
                label = np.bincount(y[mask]).argmax()
            else:
                label = -1
            self.cluster_labels_.append(label)

    def predict(self, X):
        n_samples = X.shape[0]
        n_features = X.shape[1]
        q = self.m
        V = self.V_
        F = self.F_
        A = np.zeros_like(F)
        for i in range(self.n_clusters):
            detF = np.linalg.det(F[i])
            if detF <= 0:
                detF = 1e-6
            A[i] = (detF ** (1 / n_features)) * np.linalg.inv(F[i])
        D = np.zeros((self.n_clusters, n_samples))
        for i in range(self.n_clusters):
            diff = X - V[i]
            D[i] = np.einsum('ij,jk,ik->i', diff, A[i], diff)
        clusters = np.argmin(D, axis=0)
        return np.array([self.cluster_labels_[c] for c in clusters])

    def evaluate(self, X, y_true):
        y_pred = self.predict(X)
        acc = accuracy_score(y_true, y_pred)
        cm = confusion_matrix(y_true, y_pred)
        return acc, cm

In [ ]:
clf = GKSupervisionado(n_clusters=n_clusters, random_state=42)
clf.fit(X_train, y_train)
acc, cm = clf.evaluate(X_test, y_test)
print(f'Acurácia: {acc:.4f}')
print('Matriz de Confusão:')
print(cm)
plt.figure(figsize=(6,5))
plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Matriz de Confusão - GK (Adult Balanceado)')
plt.colorbar()
plt.ylabel('Verdadeiro')
plt.xlabel('Predito')
plt.savefig('img/gk_adult_balance_confusion_matrix.png')
plt.show()

In [ ]:
acuracias = []
mse_list = []
for seed in range(1, 31):
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.3, random_state=seed, stratify=y)
    clf = GKSupervisionado(n_clusters=n_clusters, random_state=seed)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    acuracias.append(acc)
    mse_list.append(mse)
acuracias = np.array(acuracias)
mse_array = np.array(mse_list)
print(f'Acurácia média: {acuracias.mean():.4f}')
print(f'Desvio padrão da acurácia: {acuracias.std():.4f}')
print(f'MSE médio: {mse_array.mean():.4f}')
print(f'Desvio padrão do MSE: {mse_array.std():.4f}')
plt.figure(figsize=(7,4))
plt.plot(range(1, 31), acuracias, marker='o', color='tab:blue')
plt.xlabel('Repetição')
plt.ylabel('Acurácia')
plt.title('Acurácia por repetição - GK (Adult Balanceado)')
plt.tight_layout()
plt.savefig('img/gk_adult_balance_accuracy_repetitions.png')
plt.show()
plt.figure(figsize=(7,4))
plt.plot(range(1, 31), mse_array, marker='o', color='tab:red')
plt.xlabel('Repetição')
plt.ylabel('Erro Médio Quadrático (MSE)')
plt.title('MSE por repetição - GK (Adult Balanceado)')
plt.tight_layout()
plt.savefig('img/gk_adult_balance_mse_repetitions.png')
plt.show()
np.save('img/gk_adult_balance_accuracies.npy', acuracias)
np.savetxt('img/gk_adult_balance_accuracies.csv', acuracias, delimiter=',')
np.save('img/gk_adult_balance_mse_repetitions.npy', mse_array)
np.savetxt('img/gk_adult_balance_mse_repetitions.csv', mse_array, delimiter=',')

## Análise dos Resultados

Compare os resultados obtidos com o dataset balanceado e o original. Observe se houve melhora na matriz de confusão, acurácia média e estabilidade do método.